# GTFS Transport Data Exploration: 30-Minute Reachability

In this notebook, I prototype a metric to measure public transport accessibility. The goal is to calculate the **"Maximum 30-minute Reachability"** for each bus/tram stop in Krakow during morning rush hours (7:00 AM - 9:00 AM).

By tracking how far a resident can travel from a specific stop without transfers, we can objectively score the transport quality of different neighborhoods. This metric will later feed into the overall Urban Quality of Life (QoL) spatial model.

### Loading up needed modules

In [65]:
import pandas as pd
import geopandas as gpd

Before exploration, I have already extracted needed folders from a .zip file. I fetched my city borders from https://polygons.openstreetmap.fr/

In [66]:
stop_times = pd.read_csv(r"gtfs\GTFS_KRK_M\stop_times.txt")
calendar = pd.read_csv(r"gtfs\GTFS_KRK_M\calendar.txt")
trips = pd.read_csv(r"gtfs\GTFS_KRK_M\trips.txt")
stops = pd.read_csv(r"gtfs\GTFS_KRK_M/stops.txt")
routes = pd.read_csv(r"gtfs\GTFS_KRK_M/routes.txt")
borders = gpd.read_file("krakow_borders.geojson").to_crs(epsg=2180)

Quick function dedicated to converting standard time format to seconds in a day

In [67]:
def gtfs_to_seconds(time_series):
    time_parts = time_series.str.split(":", expand=True).astype(int)
    return time_parts[0] * 3600 + time_parts[1] * 60 + time_parts[2]

GTFS documentation, with every table and column name explained: https://gtfs.org/documentation/schedule/reference

My carrier does not use typical weekly schedule in GTFS data. Instead, they create different `service_id` for every change in schedule. In my tool I aim to measure a typical workweek so I will use "1582_PO" `service-id`, because it probably contains a Monday schedule

In [68]:
display(calendar)

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1582_PO,0,0,0,0,0,0,0,20260221,20270131
1,1582_SO,0,0,0,0,0,0,0,20260221,20270131
2,1582_SW,0,0,0,0,0,0,0,20260221,20270131
3,1582_PT,0,0,0,0,0,0,0,20260221,20270131
4,1582_CZ,0,0,0,0,0,0,0,20260221,20270131
5,1583_PO,0,0,0,0,0,0,0,20260221,20270131
6,1583_SO,0,0,0,0,0,0,0,20260221,20270131
7,1583_SW,0,0,0,0,0,0,0,20260221,20270131
8,1583_PT,0,0,0,0,0,0,0,20260221,20270131
9,1583_CZ,0,0,0,0,0,0,0,20260221,20270131


In [69]:
display(stop_times.head())

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
0,1582_6568,06:38:00,06:38:00,2580,1,NaN,0,1,NaN,1
1,1583_6475,06:38:00,06:38:00,2580,1,NaN,0,1,NaN,1
2,1582_6568,06:40:00,06:40:00,2590,2,NaN,0,0,NaN,1
3,1583_6475,06:40:00,06:40:00,2590,2,NaN,0,0,NaN,1
4,1582_6568,06:42:00,06:42:00,2639,3,NaN,0,0,NaN,1


In [70]:
display(routes.head())

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order,continuous_pickup,continuous_drop_off
0,100,Mobilis,100,NaN,NaN,3,NaN,FFFFFF,000000,1,NaN,NaN
1,101,Mobilis,101,NaN,NaN,3,NaN,FFFFFF,000000,2,NaN,NaN
2,103,Mobilis,103,NaN,NaN,3,NaN,FFFFFF,000000,3,NaN,NaN
3,106,Mobilis,106,NaN,NaN,3,NaN,FFFFFF,000000,4,NaN,NaN
4,108,Mobilis,108,NaN,NaN,3,NaN,FFFFFF,000000,5,NaN,NaN


In [71]:
display(stop_times.head())

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
0,1582_6568,06:38:00,06:38:00,2580,1,NaN,0,1,NaN,1
1,1583_6475,06:38:00,06:38:00,2580,1,NaN,0,1,NaN,1
2,1582_6568,06:40:00,06:40:00,2590,2,NaN,0,0,NaN,1
3,1583_6475,06:40:00,06:40:00,2590,2,NaN,0,0,NaN,1
4,1582_6568,06:42:00,06:42:00,2639,3,NaN,0,0,NaN,1


In [72]:
display(stops.head())

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,12,101-05,Stella-Sawickiego,5,50.07358,20.00343,1,NaN,0,NaN,NaN,0
1,13,101-06,Stella-Sawickiego,6,50.07487,20.00369,1,NaN,0,NaN,NaN,0
2,14,101-08,Stella-Sawickiego,8,50.07653,20.00359,1,NaN,0,NaN,NaN,0
3,17,102-03,Czyżyny,3,50.07417,20.01123,1,NaN,0,NaN,NaN,0
4,23,103-05,Rondo Czyżyńskie,5,50.07382,20.01649,1,NaN,0,NaN,NaN,0


In [73]:
display(trips.head())

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,101,1582_PO,1582_6568,Kopiec Kościuszki,NaN,0,101-01,993372,0,0
1,101,1583_PO,1583_6475,Kopiec Kościuszki,NaN,0,101-01,994809,0,0
2,100,1582_PO,1582_6572,Plac Na Stawach,NaN,1,101-01,993233,0,0
3,100,1583_PO,1583_6477,Plac Na Stawach,NaN,1,101-01,994670,0,0
4,100,1582_PO,1582_6577,Kopiec Kościuszki,NaN,0,101-01,993234,0,0


Here, I am extracting just the trips starting within my timeframe - morning high time 7:00 AM - 9:00 AM in this case

In [74]:
trip_starts = pd.merge(stop_times, trips, on="trip_id")[["trip_id", "departure_time"]]
trip_starts["departure_seconds"] = gtfs_to_seconds(trip_starts["departure_time"])
trip_starts = trip_starts.groupby("trip_id", as_index=False).agg(
    {"departure_seconds": "min"}
)
high_time_trips = trip_starts.query(
    "departure_seconds>=25200 and departure_seconds<=32400"
)["trip_id"]
high_time_trips = high_time_trips.array
high_time_trips

<NumpyExtensionArray>
['1582_10258', '1582_10261', '1582_10266', '1582_10302', '1582_10305',
 '1582_10322', '1582_10325', '1582_10328', '1582_10364', '1582_10366',
 ...
  '1583_9536',  '1583_9539',  '1583_9543',  '1583_9653',  '1583_9657',
  '1583_9661',  '1583_9983',  '1583_9987',  '1583_9991',  '1583_9995']
Length: 2878, dtype: object

Quick filtering of actually needed columns

In [75]:
stops_location = stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]]
routes_useful = routes[["route_id", "route_short_name", "route_type"]]
trips_useful = trips[["trip_id", "route_id", "service_id", "direction_id"]]
stop_times_useful = stop_times[["trip_id", "departure_time", "stop_id", "pickup_type"]]
stop_times_clean = stop_times_useful.copy()
stop_times_clean

,trip_id,departure_time,stop_id,pickup_type
0,1582_6568,06:38:00,2580,0
1,1583_6475,06:38:00,2580,0
2,1582_6568,06:40:00,2590,0
3,1583_6475,06:40:00,2590,0
4,1582_6568,06:42:00,2639,0
...,...,...,...,...
476445,1583_75822,27:35:00,77,0
476446,1582_75568,27:37:00,23,0
476447,1583_75822,27:37:00,23,0
476448,1582_75568,27:39:00,1884,1


Assigning geometry to the stops dataframe

In [76]:
stops_gdf = gpd.GeoDataFrame(
    stops_location,
    geometry=gpd.points_from_xy(stops_location["stop_lon"], stops_location["stop_lat"]),
    crs="EPSG:4326",
)
stops_gdf = stops_gdf.drop(columns=["stop_lat", "stop_lon"]).to_crs(epsg=2180)
stops_gdf

,stop_id,stop_name,geometry
0,12,Stella-Sawickiego,POINT (571780.887 245629.048)
1,13,Stella-Sawickiego,POINT (571797.56 245772.681)
2,14,Stella-Sawickiego,POINT (571787.928 245957.094)
3,17,Czyżyny,POINT (572337.966 245702.15)
4,23,Rondo Czyżyńskie,POINT (572714.761 245668.354)
...,...,...,...
790,3858,Architektów Szkoła,POINT (575836.18 248253.128)
791,3863,Sasanek,POINT (581536.355 245603.337)
792,3864,Sasanek,POINT (581536.385 245554.42)
793,3865,Wiklinowa,POINT (573427.361 243741.511)


Extracting stops within the city borders

In [77]:
stops_in_borders_map = stops_gdf.sjoin(borders, predicate="within")
stops_in_borders_array = stops_in_borders_map["stop_id"].array
stops_in_borders_array

<NumpyExtensionArray>
[  12,   13,   14,   17,   23,   24,   26,   42,   45,   50,
 ...
 3841, 3844, 3845, 3846, 3857, 3858, 3863, 3864, 3865, 3873]
Length: 794, dtype: int64

Filtering the stops so that only these within the borders remain in the data.

In [78]:
all_trips = pd.merge(stop_times_clean, trips_useful, how="left", on="trip_id")
trips_in_borders = all_trips[all_trips["stop_id"].isin(stops_in_borders_array)]
workday_trips = trips_in_borders[trips_in_borders["service_id"] == "1582_PO"].drop(
    columns="service_id"
)

In [79]:
workday_trips["departure_seconds"] = gtfs_to_seconds(workday_trips["departure_time"])
workday_trips = workday_trips.rename(columns={"stop_id": "starting_stop"}).drop(
    columns="departure_time"
)
high_time = workday_trips[workday_trips["trip_id"].isin(high_time_trips)].copy()
high_time["target_seconds"] = high_time["departure_seconds"] + 1800
target_stops = (
    high_time[["trip_id", "starting_stop", "departure_seconds"]]
    .copy()
    .rename(columns={"starting_stop": "target_stop"})
)
target_stops

,trip_id,target_stop,departure_seconds
14,1582_6572,2161,25200
16,1582_6572,2160,25260
18,1582_6572,2123,25380
20,1582_6572,2158,25440
22,1582_6572,2113,25500
...,...,...,...
108740,1582_28717,13,32460
108742,1582_28717,147,32700
108744,1582_28717,185,32820
108746,1582_28717,178,32940


### Efficient Time-Window Matching

To efficiently find the furthest stop a passenger can reach within a 30-minute window, I use Pandas' `merge_asof`. Instead of computationally expensive loops, this function performs a backward search to match the closest target time before the 30-minute limit expires.

In [80]:
high_time = high_time.sort_values("target_seconds")
target_stops = target_stops.sort_values("departure_seconds")
furthest_stops = pd.merge_asof(
    left=high_time,
    right=target_stops,
    by="trip_id",
    left_on="target_seconds",
    right_on="departure_seconds",
    direction="backward",
)
furthest_stops = furthest_stops.merge(routes_useful, how="left", on="route_id").drop(
    columns=["departure_seconds_x", "departure_seconds_y", "target_seconds"]
)
furthest_stops = furthest_stops[furthest_stops["pickup_type"] != 1].drop(
    columns="pickup_type"
)

In [ ]:
furthest_stops = (
    furthest_stops.merge(
        stops_in_borders_map, how="inner", left_on="starting_stop", right_on="stop_id"
    )
    .drop(columns="stop_id")
    .rename(
        columns={
            "stop_name": "starting_stop_name",
            "geometry": "starting_stop_location",
        }
    )
    .merge(stops_in_borders_map, how="inner", left_on="target_stop", right_on="stop_id")
    .drop(columns="stop_id")
    .rename(
        columns={"stop_name": "target_stop_name", "geometry": "target_stop_location"}
    )
)
furthest_stops = furthest_stops[
    furthest_stops["starting_stop_name"] != furthest_stops["target_stop_name"]
]
furthest_stops

,trip_id,starting_stop,route_id,direction_id,target_stop,route_short_name,route_type,starting_stop_name,starting_stop_location,index_right_x,target_stop_name,target_stop_location,index_right_y
0,1582_6572,2161,100,1,3756,100,3,Kopiec Kościuszki,POINT (563877.024 243494.395),0,Plac Na Stawach,POINT (565876.006 243497.524),0
1,1582_7034,1884,113,0,3844,113,3,Czyżyny Dworzec,POINT (572624.02 245817.2),0,Galicyjska,POINT (571949.451 243895.958),0
2,1582_14658,221,142,0,262,142,3,Os. Na Stoku,POINT (575085.797 248369.259),0,Dąbrowskiej,POINT (572559.45 246146.498),0
3,1582_1534,3585,109,0,2148,109,3,Cracovia Stadion,POINT (566025.9 243878.452),0,Bielany,POINT (558684.757 242105.38),0
4,1582_8874,3445,147,0,3712,147,3,Nowy Kleparz,POINT (566963.411 245708.808),0,Żabiniec,POINT (567736.039 246784.646),0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6189,1582_28507,170,501,1,306,501,3,Aleja Róż,POINT (574314.188 245798.192),0,Zalew Nowohucki,POINT (575216.275 245958.671),0
6190,1582_11682,152,152,0,1884,152,3,Os. Na Lotnisku,POINT (572527.39 246979.84),0,Czyżyny Dworzec,POINT (572624.02 245817.2),0
6191,1582_11682,264,152,0,1884,152,3,Os. Kościuszkowskie,POINT (572700.479 246665.357),0,Czyżyny Dworzec,POINT (572624.02 245817.2),0
6192,1582_28507,42,501,1,306,501,3,Struga,POINT (574937.265 245804.67),0,Zalew Nowohucki,POINT (575216.275 245958.671),0


In [82]:
furthest_stops_clean = gpd.GeoDataFrame(
    furthest_stops[
        [
            "route_short_name",
            "route_type",
            "direction_id",
            "starting_stop_name",
            "starting_stop_location",
            "target_stop_name",
            "target_stop_location",
        ]
    ],
    geometry="starting_stop_location",
    crs="EPSG:2180",
)
furthest_stops_clean["max_reach_km"] = (
    furthest_stops_clean.distance(furthest_stops_clean["target_stop_location"]) / 1000
)
stop_reach = (
    furthest_stops_clean.groupby(
        [
            "starting_stop_name",
            "starting_stop_location",
            "route_short_name",
            "direction_id",
            "route_type",
        ],
        as_index=False,
    )
    .agg({"max_reach_km": "mean"})
    .rename(
        columns={
            "starting_stop_name": "stop_name",
            "starting_stop_location": "stop_location",
            "route_short_name": "route_number",
        }
    )
)
final_reachability = (
    stop_reach.groupby(
        ["stop_location", "stop_name", "route_number", "direction_id", "route_type"],
        as_index=False,
    )
    .agg({"max_reach_km": "sum"})
    .sort_values(by="max_reach_km", ascending=False)
)
final_reachability_gdf = gpd.GeoDataFrame(
    final_reachability, geometry="stop_location", crs="EPSG:2180"
)
final_reachability_gdf

,stop_location,stop_name,route_number,direction_id,route_type,max_reach_km
773,POINT (571605.396 247138.603),Wiślicka,578,0,3,11.087178
163,POINT (563681.867 239383.397),Czerwone Maki P+R,578,1,3,10.328960
849,POINT (571955.16 247775.861),Łęczycka,578,0,3,10.317424
192,POINT (563767.354 239302.153),Czerwone Maki P+R,578,1,3,10.312708
738,POINT (571780.887 245629.048),Stella-Sawickiego,578,0,3,10.227526
...,...,...,...,...,...,...
671,POINT (573768.726 245974.066),Os. Zgody,502,0,3,0.206952
1086,POINT (573079.288 239196.578),Ćwiklińskiej,133,0,3,0.203179
288,POINT (564190.843 247933.688),Piaskowa,501,0,3,0.182659
696,POINT (572714.761 245668.354),Rondo Czyżyńskie,113,0,3,0.174324


### Conclusion & Production Pipeline

*Note: This notebook serves as a Proof of Concept (PoC) for a single transport carrier to validate the mathematical logic. The production-ready code—which parameterizes these time windows, scales the process across all city carriers, and integrates with the overall scoring model—can be found in `src/assemble_transport_data.py`.*